In [ ]:
!pip install pdfplumber nltk gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 81.9 MB/s eta 0:00:00


In [ ]:
import pdfplumber
import nltk
import numpy as np
import pandas as pd

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
nltk.download('punkt_tab',quiet=True)
nltk.download('stopwords',quiet=True)
nltk.download('wordnet',quiet=True)
print("Success")

Success


In [ ]:
def extract_text_from_pdf(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + " "

    return text

In [ ]:
pdf_text = extract_text_from_pdf("/content/II-II-ARTIFICIAL INTELLIGENCE-LECTURE NOTES.pdf")

print(pdf_text[:1000])

SVR ENGINEERING COLLEGE ,ARTIFICIAL INTELLIGENCE MATERIAL
ARTIFICIAL
INTELLIGENCE
1
SVR ENGINEERING COLLEGE ,ARTIFICIAL INTELLIGENCE MATERIAL SVR ENGINEERING COLLEGE ,ARTIFICIAL INTELLIGENCE MATERIAL
UNIT-I
Syllabus
Introduction: What is AI, Foundations of AI, History of AI, The State of Art.
Intelligent Agents: Agents and Environments, Good Behavior: The Concept of Rationality,
The Nature of Environments, The Structure of Agents.
2
SVR ENGINEERING COLLEGE ,ARTIFICIAL INTELLIGENCE MATERIAL SVR ENGINEERING COLLEGE ,ARTIFICIAL INTELLIGENCE MATERIAL
CHAPTER-1 INTRODUCTION
 The field of artificial intelligence, or AI, goes further still: it attempts not just to understand but also
to build intelligent entities.
 AI is one of the newest sciences. Work started in earnest soon after World War II, and the name
itself was coined in 1956.
 Along with molecular biology, AI is regularly cited as the "field I would most like to be in" by
scientists in other disciplines.
 A student in physics mi

In [ ]:
paragraphs = pdf_text.split("\n")

paragraphs = [p for p in paragraphs if len(p) > 100]

print("Total paragraphs:", len(paragraphs))

Total paragraphs: 1006


In [ ]:
import re
stop_words = set(stopwords.words('english'))

def preprocess(text):

    tokens = word_tokenize(text.lower())
    text = re.sub(r'https?://\S+|www\.\S+|jntua', '', text, flags=re.IGNORECASE)
    text = re.sub(r'[^a-z\s]', '', text)
    cleaned = []

    for word in tokens:
        if word.isalpha() and word not in stop_words:
            cleaned.append(word)

    return cleaned

In [ ]:
processed_text = []

for para in paragraphs:
    processed_text.append(preprocess(para))

In [ ]:
processed_text

[['svr',
  'engineering',
  'college',
  'artificial',
  'intelligence',
  'material',
  'svr',
  'engineering',
  'college',
  'artificial',
  'intelligence',
  'material'],
 ['svr',
  'engineering',
  'college',
  'artificial',
  'intelligence',
  'material',
  'svr',
  'engineering',
  'college',
  'artificial',
  'intelligence',
  'material'],
 ['field',
  'artificial',
  'intelligence',
  'ai',
  'goes',
  'still',
  'attempts',
  'understand',
  'also'],
 ['galileo',
  'newton',
  'einstein',
  'rest',
  'ai',
  'hand',
  'still',
  'openings',
  'several'],
 ['ai',
  'systematizes',
  'automates',
  'intellectual',
  'tasks',
  'therefore',
  'potentially',
  'relevant',
  'sphere'],
 ['svr',
  'engineering',
  'college',
  'artificial',
  'intelligence',
  'material',
  'svr',
  'engineering',
  'college',
  'artificial',
  'intelligence',
  'material'],
 ['several',
  'greek',
  'schools',
  'developed',
  'various',
  'forms',
  'logic',
  'notation',
  'rules',
  'derivation

In [ ]:
model = Word2Vec(
    sentences=processed_text,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

In [ ]:
def paragraph_vector(tokens):

    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [ ]:
paragraph_vectors = []

for tokens in processed_text:
    vec = paragraph_vector(tokens)
    paragraph_vectors.append(vec)

In [ ]:
similarity_matrix = cosine_similarity(paragraph_vectors)

In [ ]:
def recommend_similar_paragraph(index, top_n=3):

    scores = list(enumerate(similarity_matrix[index]))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    scores = scores[1:top_n+1]

    print("\nSelected Paragraph:\n")
    print(paragraphs[index])

    print("\nRecommended Similar Paragraphs:\n")

    for i in scores:
        print("Similarity:", round(i[1],2))
        print(paragraphs[i[0]])
        print()

In [ ]:
recommend_similar_paragraph(89)


Selected Paragraph:

 First, we need some information about how the world evolves independently of the agent-for examp le,

Recommended Similar Paragraphs:

Similarity: 0.93
 Second, we need some information about how the agent's own actions affect the world-for example, that

Similarity: 0.93
second. The executive layer is also responsible for integrating sensor information into an internal state

Similarity: 0.92
 Doing actions in order to modify future percepts-sometimes called information gathering-is an important part

